In [97]:
!uv pip install torch transformers accelerate pandas || pip install torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F
import pandas as pd

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('mps')
print(f"Using device: {device}")

Checked 4 packages in 17ms
Using device: mps


In [98]:
tokenizer = AutoTokenizer.from_pretrained("SamLowe/roberta-base-go_emotions")
model = AutoModelForSequenceClassification.from_pretrained("SamLowe/roberta-base-go_emotions", device_map="auto")

train_df = pd.read_csv("data/train.csv")
models = {name: i for i, name in enumerate(sorted(train_df["model"].unique()))}
train_df["model_class"] = train_df["model"].map(models)

# Keep token IDs rectangular and within the model's supported sequence length.
encoded_text = tokenizer(
    train_df["text"].fillna("").tolist(),
    padding=True,
    truncation=True,
    max_length=tokenizer.model_max_length,
    return_tensors="pt",
)
train_df["text_tokens"] = encoded_text["input_ids"].tolist()
train_df["attention_mask"] = encoded_text["attention_mask"].tolist()

# One-hot encode model labels; token IDs belong to the tokenizer vocabulary.
y_train = torch.tensor(train_df["model_class"].to_numpy(), dtype=torch.long)
y_train_ohe = F.one_hot(y_train, num_classes=len(models)).float()
train_df["model_ohe"] = y_train_ohe.tolist()

train_df

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 63207.01it/s]


,id,prompt_id,text,model,model_class,text_tokens,attention_mask,model_ohe
0,0018b3765972,p081,"When a language dies, a particular astronomy d...",qwen3.8-max,17,"[0, 1779, 10, 2777, 8524, 6, 10, 1989, 36095, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,0041ac86ee93,p073,Data science communities matter because they a...,claude-sonnet-5,4,"[0, 30383, 2866, 1822, 948, 142, 51, 10107, 22...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, ..."
2,005a19083eae,p120,"In the final twilight of time, the last embers...",gemini-3.6-flash,6,"[0, 1121, 5, 507, 38262, 9, 86, 6, 5, 94, 2841...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, ..."
3,009319596f10,p073,Data science communities matter because they t...,gpt-5.6-terra,14,"[0, 30383, 2866, 1822, 948, 142, 51, 1004, 10,...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,00d3e8c92596,p097,"Deep inside each ear, three tiny loops of bone...",glm-5.3-flash,10,"[0, 35166, 1025, 349, 5567, 6, 130, 5262, 3748...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
...,...,...,...,...,...,...,...,...
2694,ff4344ceda23,p075,"Across centuries of silence and shadow, an inv...",gemini-3.8-flash,8,"[0, 42945, 11505, 9, 7308, 8, 9099, 6, 41, 207...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, ..."
2695,ff507658ba82,p092,Though both instruments breathe the exact same...,gemini-3.7-flash,7,"[0, 26223, 258, 9571, 14575, 5, 6089, 276, 645...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, ..."
2696,ff50c252810c,p113,A single photon leaves the star as neither wav...,claude-opus-4-8,2,"[0, 250, 881, 44635, 3607, 5, 999, 25, 5063, 4...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2697,ffcf6321f7e9,p073,Data science communities matter because they t...,glm-5.3,9,"[0, 30383, 2866, 1822, 948, 142, 51, 7891, 806...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


In [99]:
print(train_df["text_tokens"].iloc[0])

X_train = encoded_text["input_ids"]
attention_mask_train = encoded_text["attention_mask"]
# y_train holds class indices; y_train_ohe holds one-hot labels.
train_df[["text_tokens", "model_class", "model_ohe"]]

[0, 1779, 10, 2777, 8524, 6, 10, 1989, 36095, 14548, 29, 35, 45, 129, 2523, 13, 2690, 6, 53, 1086, 25187, 119, 2726, 9, 1503, 14, 5850, 6864, 141, 7, 1166, 5, 363, 4, 993, 39279, 5456, 5, 6360, 30, 41490, 4128, 6, 17014, 5, 36713, 4846, 7, 22519, 6, 12323, 6, 50, 5, 8702, 14561, 9, 7723, 131, 643, 46855, 21641, 17369, 25, 47041, 6, 98, 86, 16, 45, 10, 516, 53, 10, 16331, 8016, 4, 83, 2136, 64, 2324, 5, 6089, 3195, 9, 14131, 15, 1958, 6, 5, 35630, 9, 2686, 7, 4775, 6, 50, 5, 25634, 31605, 88, 17839, 32895, 8, 11651, 999, 12201, 4, 590, 23989, 6, 167, 3115, 6277, 88, 7308, 4, 440, 36451, 64, 36839, 5, 6360, 167, 6864, 36308, 6, 3486, 5, 5128, 10250, 24, 5850, 4, 166, 2217, 43721, 14, 156, 5, 9468, 11179, 6, 13557, 6, 8, 4299, 6, 12793, 119, 20643, 11, 61, 15073, 16, 10, 30443, 6, 3137, 2580, 32, 8383, 2584, 30, 6774, 6, 8, 10759, 1641, 1635, 32, 45, 4460, 3493, 53, 10175, 9582, 4, 20, 872, 16, 45, 8315, 36912, 3569, 131, 24, 16, 5, 3335, 24669, 9, 678, 1050, 5274, 7, 5170, 4, 653, 1189, 

,text_tokens,model_class,model_ohe
0,"[0, 1779, 10, 2777, 8524, 6, 10, 1989, 36095, ...",17,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,"[0, 30383, 2866, 1822, 948, 142, 51, 10107, 22...",4,"[0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, ..."
2,"[0, 1121, 5, 507, 38262, 9, 86, 6, 5, 94, 2841...",6,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, ..."
3,"[0, 30383, 2866, 1822, 948, 142, 51, 1004, 10,...",14,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,"[0, 35166, 1025, 349, 5567, 6, 130, 5262, 3748...",10,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
...,...,...,...
2694,"[0, 42945, 11505, 9, 7308, 8, 9099, 6, 41, 207...",8,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, ..."
2695,"[0, 26223, 258, 9571, 14575, 5, 6089, 276, 645...",7,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, ..."
2696,"[0, 250, 881, 44635, 3607, 5, 999, 25, 5063, 4...",2,"[0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2697,"[0, 30383, 2866, 1822, 948, 142, 51, 7891, 806...",9,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


In [100]:
class TextClassifierHead(torch.nn.Module):
    def __init__(self, in_features=768, out_features=2, dropout=0.1):
        super().__init__()
        if out_features < 1:
            raise ValueError("out_features must be at least 1")
        self.in_features = in_features
        self.out_features = out_features
        self.dropout = torch.nn.Dropout(dropout)
        self.dense = torch.nn.Linear(in_features, in_features)
        self.out_proj = torch.nn.Linear(in_features, out_features)

    def forward(self, features, **kwargs):
        # RoBERTa passes hidden states with shape (batch, sequence, hidden).
        x = self.dropout(features[:, 0, :])
        x = torch.tanh(self.dense(x))
        return self.out_proj(self.dropout(x))


In [101]:
# Replace the emotion classifier with a trainable model-label classifier.
old_head_parameter = next(model.classifier.parameters())
dropout = model.config.classifier_dropout
if dropout is None:
    dropout = model.config.hidden_dropout_prob

new_head = TextClassifierHead(
    in_features=model.config.hidden_size,
    out_features=len(models),
    dropout=dropout,
)
new_head.apply(model._init_weights)
model.classifier = new_head.to(
    device=old_head_parameter.device,
    dtype=old_head_parameter.dtype,
)

model.num_labels = len(models)
model.config.num_labels = len(models)
model.config.label2id = dict(models)
model.config.id2label = {index: name for name, index in models.items()}
model.config.problem_type = "single_label_classification"

# Pass integer y_train labels to model(..., labels=...) for cross-entropy loss.
model.classifier


TextClassifierHead(
  (dropout): Dropout(p=0.1, inplace=False)
  (dense): Linear(in_features=768, out_features=768, bias=True)
  (out_proj): Linear(in_features=768, out_features=18, bias=True)
)

In [102]:
from pathlib import Path
from torch.utils.data import DataLoader, TensorDataset, Subset

seed = 42
batch_size = 8
num_epochs = 3
learning_rate = 2e-5
weight_decay = 0.01
validation_fraction = 0.2
checkpoint_path = Path("checkpoints/best_classifier.pt")

torch.manual_seed(seed)

# Hold out entire prompts to prevent prompt overlap between train and validation.
prompt_ids = train_df["prompt_id"].drop_duplicates().sample(frac=1, random_state=seed)
if len(prompt_ids) < 2:
    raise ValueError("At least two distinct prompts are needed for validation")
num_val_prompts = max(1, min(len(prompt_ids) - 1, round(len(prompt_ids) * validation_fraction)))
is_validation = train_df["prompt_id"].isin(prompt_ids.iloc[:num_val_prompts])
train_indices = torch.tensor((~is_validation).to_numpy()).nonzero(as_tuple=True)[0].tolist()
val_indices = torch.tensor(is_validation.to_numpy()).nonzero(as_tuple=True)[0].tolist()

dataset = TensorDataset(X_train, attention_mask_train, y_train)
train_loader = DataLoader(
    Subset(dataset, train_indices), batch_size=batch_size, shuffle=True,
    generator=torch.Generator().manual_seed(seed),
)
val_loader = DataLoader(Subset(dataset, val_indices), batch_size=batch_size)

# Respect the device selected when the pretrained model was loaded.
device = model.get_input_embeddings().weight.device
optimizer = torch.optim.AdamW(
    (p for p in model.parameters() if p.requires_grad),
    lr=learning_rate, weight_decay=weight_decay,
)
print(f"Device: {device} | Training: {len(train_indices)} | Validation: {len(val_indices)}")


Device: mps:0 | Training: 2159 | Validation: 540


In [103]:
from tqdm.auto import tqdm

def run_epoch(model, loader, device, optimizer=None, description="Training"):
    training = optimizer is not None
    model.train(training)
    total_loss, total_correct, total_examples = 0.0, 0, 0

    with torch.set_grad_enabled(training):
        progress = tqdm(loader, desc=description, unit="batch", leave=True)
        for input_ids, attention_mask, labels in progress:
            # Trim trailing padding to the longest text in this batch.
            sequence_length = int(attention_mask.sum(dim=1).max().item())
            input_ids = input_ids[:, :sequence_length].to(device)
            attention_mask = attention_mask[:, :sequence_length].to(device)
            labels = labels.to(device)

            if training:
                optimizer.zero_grad(set_to_none=True)

            outputs = model(
                input_ids=input_ids, attention_mask=attention_mask, labels=labels,
            )
            loss = outputs.loss
            if not torch.isfinite(loss):
                raise RuntimeError("Non-finite loss encountered during training/validation")

            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            count = labels.size(0)
            total_loss += loss.item() * count
            total_correct += (outputs.logits.argmax(dim=-1) == labels).sum().item()
            total_examples += count
            progress.set_postfix(
                loss=f"{total_loss / total_examples:.4f}",
                accuracy=f"{total_correct / total_examples:.2%}",
                refresh=True,
            )

    return total_loss / total_examples, total_correct / total_examples


history = []
best_val_loss = float("inf")
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)

for epoch in range(1, num_epochs + 1):
    print(f"\nEpoch {epoch}/{num_epochs}", flush=True)
    train_loss, train_accuracy = run_epoch(
        model, train_loader, device, optimizer,
        description=f"Epoch {epoch}/{num_epochs} train",
    )
    val_loss, val_accuracy = run_epoch(
        model, val_loader, device,
        description=f"Epoch {epoch}/{num_epochs} validation",
    )
    history.append({
        "epoch": epoch, "train_loss": train_loss, "train_accuracy": train_accuracy,
        "val_loss": val_loss, "val_accuracy": val_accuracy,
    })
    print(
        f"Epoch {epoch}/{num_epochs} | "
        f"train loss {train_loss:.4f}, accuracy {train_accuracy:.2%} | "
        f"val loss {val_loss:.4f}, accuracy {val_accuracy:.2%}",
        flush=True,
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        # Include configuration and label mapping for the custom classifier head.
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "config": model.config.to_dict(),
            "label2id": dict(models),
            "val_loss": val_loss,
        }, checkpoint_path)
        tokenizer.save_pretrained(checkpoint_path.parent)
        print(f"Saved best checkpoint to {checkpoint_path}")

history_df = pd.DataFrame(history)
history_df



Epoch 1/3


Epoch 1/3 train:  17%|█▋        | 47/270 [00:54<04:18,  1.16s/batch, accuracy=6.38%, loss=2.9177]


KeyboardInterrupt: 